<a href="https://colab.research.google.com/github/Chathu283/Statistical-Learning-e23218/blob/main/Bayesian_Inference_Assignment_Answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses



### 1. Visualizing the Mechanics
The two-parameter logistic (2PL) item response model defines the probability of a correct response given a latent ability $\theta$ as:
$$p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$

* **Effect of $b_i$ (Difficulty parameter):** Shifting $b_i$ translates the logistic curve horizontally along the $\theta$-axis. The value of $b_i$ corresponds exactly to the point where $p_i(\theta) = 0.5$. If $b_i$ increases, the curve shifts to the **right**, meaning a user requires a higher latent ability level to achieve a $50\%$ probability of answering the item correctly.
* **Effect of $a_i$ (Discrimination parameter):** Modifies the slope of the curve at $\theta = b_i$. A larger $a_i$ creates a steeper, sharper transition between an incorrect and correct probability, making the item highly sensitive to ability changes around $\theta = b_i$.

---

### 2. Sequential Likelihood Contribution
For an isolated response $y_k \in \{0, 1\}$ at step $k$:
$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} = \left(\frac{1}{1 + e^{-a_k(\theta - b_k)}}\right)^{y_k} \left(\frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}}\right)^{1 - y_k}$$

Assuming local independence between items conditional on $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, \dots, y_k)$ is the product of individual item likelihoods:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

---

### 3. Mathematical Formulation of the Running Update
Using Bayes' Theorem under a sequential framework, the posterior distribution at step $k-1$ becomes the prior distribution for step $k$. Up to a proportionality constant:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \frac{1}{1 + e^{-a_k(\theta - b_k)}} \right]^{y_k} \left[ \frac{e^{-a_k(\theta - b_k)}}{1 + e^{-a_k(\theta - b_k)}} \right]^{1 - y_k} \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

---

### 4. Dynamic Shifting Mechanics
When $y_k = 1$ for an item with a high difficulty $b_k$, the likelihood contribution is $p_k(\theta) = \frac{1}{1 + e^{-a_k(\theta - b_k)}}$.
* Because $b_k$ is large, this curve remains very close to $0$ for low values of $\theta$ and climbs toward $1$ only when $\theta \ge b_k$.
* Multiplying the prior by this heavily asymmetric, right-leaning likelihood suppresses the mass of the distribution on the left (lower abilities) and dramatically amplifies the right tail.
* Consequently, the product shifts the peak (mode) and the center of mass of the running posterior strongly toward the **right** (higher values of $\theta$).

---

### 5. Tracking Certainty and Sharpness
The discrimination parameter $a_k$ controls how much information the item provides about $\theta$.
* **Very Large $a_k$:** The likelihood contribution approaches a step function at $\theta = b_k$. Multiplying the prior by this step function acts as a sharp filter, carving off half of the distribution and localizing the remaining probability mass. This causes a dramatic reduction in posterior variance (creating a sharp, narrow peak), maximizing the platform's certainty increase.
* **Very Small $a_k$:** The likelihood contribution is nearly flat across the entire $\theta$ spectrum. Multiplying the prior by an almost flat line scales its amplitude down uniformly but leaves its shape, peak, and variance virtually unchanged, adding negligible information.

---

### 6. Numerical Implementation of a Running Grid
To implement this numerically on a computer:
1. Define a fixed, uniformly spaced discrete grid vector $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ spanning a reasonable physical domain, e.g., $[-4, 4]$ with a fine step $\Delta \theta$.
2. Initialize the prior vector using the standard normal PDF: $\mathbf{f}^{(0)} = \frac{1}{\sqrt{2\pi}} \exp(-\boldsymbol{\theta}^2 / 2)$. Normalizing initially guarantees $\sum_{m=1}^M f^{(0)}_m \Delta \theta = 1$.
3. For each sequential step $k$ where a response $y_k$ is registered:
   * Evaluate the vector of likelihood values $\mathbf{L}_k = [L(y_k \mid \theta_1), \dots, L(y_k \mid \theta_M)]$.
   * Perform an element-wise multiplication to get the unnormalized posterior: $\mathbf{f}_{\text{raw}}^{(k)} = \mathbf{L}_k \odot \mathbf{f}^{(k-1)}$.
   * Compute the normalization integral constant using the trapezoidal rule: $I = \text{np.trapezoid}(\mathbf{f}_{\text{raw}}^{(k)}, \boldsymbol{\theta})$.
   * Compute the clean, normalized posterior vector: $\mathbf{f}^{(k)} = \frac{\mathbf{f}_{\text{raw}}^{(k)}}{I}$.

In [2]:
import numpy as np
import plotly.graph_objects as go

# Seed for reproducibility
np.random.seed(42)

# 1. Setup Simulation Environment
theta_true = 0.75
n_items = 20

# Define a fine grid for theta
grid_size = 1000
theta_grid = np.linspace(-4, 4, grid_size)
delta_theta = theta_grid[1] - theta_grid[0]

# Generate random item parameters
b = np.random.normal(0, 1, n_items)        # Difficulty
a = np.random.uniform(0.5, 2.0, n_items)   # Discrimination

# Generate true item response probabilities and simulate user answers
p_true = 1 / (1 + np.exp(-a * (theta_true - b)))
y = (np.random.uniform(0, 1, n_items) < p_true).astype(int)

# 2. Sequential Inference Loop
# Initialize with standard normal prior
posterior = (1 / np.sqrt(2 * np.pi)) * np.exp(-theta_grid**2 / 2)
posterior /= np.trapezoid(posterior, theta_grid)

# Storage for point estimators
theta_bayes_history = [0.0]  # Prior mean is 0
theta_map_history = [0.0]    # Prior MAP is 0

for k in range(n_items):
    # Calculate single item likelihood contribution over the grid
    p_k = 1 / (1 + np.exp(-a[k] * (theta_grid - b[k])))
    likelihood = (p_k ** y[k]) * ((1 - p_k) ** (1 - y[k]))

    # Bayesian recursive update
    unnormalized_posterior = likelihood * posterior
    posterior = unnormalized_posterior / np.trapezoid(unnormalized_posterior, theta_grid)

    # Calculate Running Estimators
    bayes_est = np.trapezoid(theta_grid * posterior, theta_grid)
    map_est = theta_grid[np.argmax(posterior)]

    theta_bayes_history.append(bayes_est)
    theta_map_history.append(map_est)

# 3. Interactive Visualization using Plotly
steps = np.arange(0, n_items + 1)
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=theta_bayes_history, mode='lines+markers', name='Posterior Mean (Bayes)'))
fig.add_trace(go.Scatter(x=steps, y=theta_map_history, mode='lines+markers', name='Maximum A Posteriori (MAP)'))
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*len(steps), mode='lines', line=dict(dash='dash', color='red'), name='True Ability (0.75)'))

fig.update_layout(
    title='Sequential Bayesian Estimation of Latent Ability Parameter',
    xaxis_title='Item Sequence Step (k)',
    yaxis_title='Estimated Latent Ability (theta)',
    template='plotly_white'
)
fig.show()

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates



### 1. Structural Probability and Properties
* **$(\alpha=1, \beta=1)$**: Represents a uniform probability density over $[0, 1]$. The center of mass sits perfectly at $0.5$, indicating complete baseline uncertainty.
* **$(\alpha=2, \beta=8)$**: The density function is skewed to the right, concentrating its mass near the left boundary with a peak (mode) at $\frac{2-1}{2+8-2} = 0.125$.
* **$(\alpha=8, \beta=2)$**: The density function is skewed to the left, concentrating mass toward the right boundary with a peak at $\frac{8-1}{8+2-2} = 0.875$.
* **Interpretation**: The parameter $\alpha$ acts as a count of virtual "successes" (clicks) and $\beta$ acts as a count of "failures" (non-clicks). Adjusting their balance shifts the distribution's center of mass toward the domain boundary that matches the dominant parameter.

---

### 2. Sequential Likelihood and Joint History
For an isolated interaction $y_k \in \{0, 1\}$ at step $k$:
$$L(y_k \mid \theta) = \theta^{y_k}(1 - \theta)^{1 - y_k}$$

For a running history vector of independent trials $\mathbf{y}^{(k)} = (y_1, \dots, y_k)$:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i}(1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

---

### 3. Closed-Form Analytical Updates (Conjugacy Proof)
Using Bayes' theorem for the step-$k$ update:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

Substituting the Bernoulli likelihood and the previous Beta prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) = \text{Beta}(\alpha_{k-1}, \beta_{k-1})$:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{y_k}(1-\theta)^{1-y_k} \right] \cdot \left[ \theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1} \right]$$
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\\beta_{k-1} + 1 - y_k) - 1}$$

This functional form matches the kernel of a Beta distribution exactly. Thus, by induction, the posterior remains within the Beta family. The closed-form analytical updates are:
$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$

The exact expression for the expectation (Posterior Mean) at step $k$ is:
$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

---

### 4. Dynamic Shifting Mechanics
* **Click ($y_k = 1$):** Directly increments $\alpha_k$ while keeping $\beta_k$ constant. This mathematically shifts the center of mass and the peak of the density function toward the right (closer to $1$).
* **Non-click ($y_k = 0$):** Directly increments $\beta_k$ while keeping $\alpha_k$ constant, shifting the density curve toward the left (closer to $0$).
* **Comparison:** In this conjugate Beta-Binomial framework, updates are computed instantly using simple arithmetic additions. This avoids the need for computationally heavy grid-based numerical integration loops, which are strictly required for non-conjugate architectures like the 2PL IRT model.

---

### 5. Running Point Estimators
Given the updated parameters $\alpha_k$ and $\beta_k$, the exact closed-form point estimators are:
* **Running Posterior Mean:**
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$
* **Running Maximum A Posteriori (MAP):**
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad (\text{for } \alpha_k > 1, \beta_k > 1)$$

In [4]:
import numpy as np
import plotly.graph_objects as go

# Seed for reproducibility
np.random.seed(42)

# Parameters
theta_true = 0.35
n_impressions = 100

# Initialize prior states
alpha_k = 1
beta_k = 1

# Storage logs
bayes_history = [alpha_k / (alpha_k + beta_k)]
map_history = [0.5]

# Generate simulated random user events
y = (np.random.uniform(0, 1, n_impressions) < theta_true).astype(int)

# Analytical updates execution
for k in range(n_impressions):
    alpha_k += y[k]
    beta_k += (1 - y[k])

    # Compute point estimators
    bayes_est = alpha_k / (alpha_k + beta_k)
    map_est = (alpha_k - 1) / (alpha_k + beta_k - 2) if (alpha_k > 1 and beta_k > 1) else 0.5

    bayes_history.append(bayes_est)
    map_history.append(map_est)

# Plotly tracking chart
steps = np.arange(0, n_impressions + 1)
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=bayes_history, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=steps, y=map_history, mode='lines', name='Posterior Mode (MAP)'))
fig.add_trace(go.Scatter(x=steps, y=[theta_true]*len(steps), mode='lines', line=dict(dash='dot', color='black'), name='True CTR (0.35)'))

fig.update_layout(
    title='Sequential Beta-Binomial Update Timeline for Ad CTR',
    xaxis_title='Impressions Milestone (k)',
    yaxis_title='Estimated Click-Through Rate',
    template='plotly_white'
)
fig.show()

# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates



### 1. Prior Belief Boundaries
For $\Theta \sim \text{Beta}(8, 1.5)$, the expected prior stiffness efficiency is calculated as:
$$\mathbb{E}[\Theta^{(0)}] = \frac{\alpha_0}{\alpha_0 + \beta_0} = \frac{8}{8 + 1.5} = \frac{8}{9.5} \approx 0.842$$

This highly asymmetric distribution places its probability mass near $\theta = 1.0$. This makes it an ideal prior for engineering components assumed to be healthy, as it incorporates the structural knowledge that an active aerospace or structural component is highly unlikely to be deployed in a severely degraded state.

---

### 2. Structural Likelihood Formulation
The physics measurement model states $y_k = \theta K_{\text{nominal}} e^{\epsilon_k}$ with $\epsilon_k \sim \mathscr{N}(0, \sigma^2)$. Rearranging for the error term gives:
$$\epsilon_k = \log\left(\frac{y_k}{\theta K_{\text{nominal}}}\right) = \log(y_k) - \log(\theta) - \log(K_{\text{nominal}})$$

Using the transformation properties of log-normal variables, the likelihood contribution of a single continuous sensor reading $y_k$ is:
$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\log(y_k) - \log(\theta) - \log(K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

Assuming independent measurement errors, the joint likelihood for the history vector $\mathbf{y}^{(k)}$ is:
$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\log(y_i) - \log(\theta) - \log(K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$

---

### 3. Mathematical Formulation of the Non-Conjugate Grid Update
An exact analytical solution does not exist because combining a polynomial-based Beta prior kernel $\theta^{\alpha-1}(1-\theta)^{\beta-1}$ with a transcendental log-normal likelihood component $\exp(-(\log y_k - \log \theta - \log K)^2 / 2\sigma^2)$ creates an algebraic form whose integrating constant cannot be evaluated in closed form.

The recursive mathematical relationship up to a proportionality constant is:
$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \frac{1}{\theta} \exp\left( -\frac{\left(\log(y_k) - \log(\theta) - \log(K_{\text{nominal}})\right)^2}{2\sigma^2} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

---

### 4. Running Point Estimates
* **Running Posterior Mean:**
$$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\int_{0}^{1} \theta \cdot f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}{\int_{0}^{1} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta}$$

* **Running Maximum A Posteriori (MAP):**
$$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \operatorname{arg\,max}_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$$

---

### 5. Algorithmic Grid Approximation and Normalization
1. Generate a line grid vector $\boldsymbol{\theta} = [\theta_1, \dots, \theta_M]$ from $0.01$ to $1.0$.
2. Compute the prior density vector over the grid points: $\mathbf{f}^{(0)} \propto \boldsymbol{\theta}^{\alpha_0-1}(1-\boldsymbol{\theta})^{\beta_0-1}$. Normalize using `np.trapezoid(f, theta_grid)`.
3. When a new sensor measurement $y_k$ arrives:
   * Evaluate the likelihood vector across all grid coordinates: $\mathbf{L}_m = \frac{1}{\theta_m} \exp\left(-\frac{(\log(y_k) - \log(\theta_m) - \log(K_{\text{nominal}}))^2}{2\sigma^2}\right)$.
   * Update the unnormalized posterior array: $\mathbf{f}_{\text{raw}}^{(k)} = \mathbf{L} \odot \mathbf{f}^{(k-1)}$.
   * Calculate the normalizing integral via the trapezoidal rule: $I = \sum_{m=1}^{M-1} \frac{f_{\text{raw}, m}^{(k)} + f_{\text{raw}, m+1}^{(k)}}{2} \Delta \theta$.
   * Normalize the distribution vector: $\mathbf{f}^{(k)} = \frac{\mathbf{f}_{\text{raw}}^{(k)}}{I}$.

In [7]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Simulation Constants
np.random.seed(10)
theta_true = 0.68
K_nominal = 50.0
sigma = 0.15
n_steps = 15

# Set up discrete bounded grid infrastructure
theta_grid = np.linspace(0.01, 1.0, 1000)

# 1. Initialize Prior State Vector
posterior = beta.pdf(theta_grid, 8, 1.5)
posterior /= np.trapezoid(posterior, theta_grid)

# Simulation logs
bayes_history = [np.trapezoid(theta_grid * posterior, theta_grid)]
map_history = [theta_grid[np.argmax(posterior)]]
milestones = {0: posterior.copy()}

# Generate simulated sensor responses
epsilon = np.random.normal(0, sigma, n_steps)
y_sensor = theta_true * K_nominal * np.exp(epsilon)

# 2. Execution Loop
for k in range(1, n_steps + 1):
    y_k = y_sensor[k-1]

    # Non-conjugate likelihood vector calculations
    log_diff = np.log(y_k) - np.log(theta_grid) - np.log(K_nominal)
    likelihood = (1.0 / theta_grid) * np.exp(-(log_diff**2) / (2 * sigma**2))

    # Integration step
    unnormalized = likelihood * posterior
    posterior = unnormalized / np.trapezoid(unnormalized, theta_grid)

    # Store milestone states for tracking
    if k in [1, 2, 5, 10, 15]:
        milestones[k] = posterior.copy()

    # Calculate point metrics
    bayes_history.append(np.trapezoid(theta_grid * posterior, theta_grid))
    map_history.append(theta_grid[np.argmax(posterior)])

# 3. Visualizations
# Plot 1: Full Posterior Density Evolution
fig1 = go.Figure()
for milestone, dist in milestones.items():
    fig1.add_trace(go.Scatter(x=theta_grid, y=dist, mode='lines', name=f'Step k={milestone}'))
fig1.update_layout(title='Evolution of Bounded Posterior Density Curves', xaxis_title='Stiffness Efficiency Factor', template='plotly_white')
fig1.show()

# Plot 2: Convergence Tracking Timeline
fig2 = go.Figure()
steps = np.arange(0, n_steps + 1)
fig2.add_trace(go.Scatter(x=steps, y=bayes_history, name='Posterior Mean'))
fig2.add_trace(go.Scatter(x=steps, y=map_history, name='MAP Estimate'))
fig2.add_trace(go.Scatter(x=steps, y=[theta_true]*len(steps), line=dict(dash='dash', color='red'), name='True Stiffness (0.68)'))
fig2.update_layout(title='Point Estimator Convergence Timeline', xaxis_title='Inspection Step', yaxis_title='Stiffness Factor', template='plotly_white')
fig2.show()

# Q. Gaussian Mixture Clustering as Conditional Updating



### 1. Deriving the Marginal Density
By the Law of Total Probability, the marginal density $p(x_i)$ is found by summing the joint distribution over all possible discrete latent categories $K$:
$$p(x_i) = \sum_{k=1}^K p(x_i, C_i = k) = \sum_{k=1}^K P(C_i = k) p(x_i \mid C_i = k)$$

Substituting the model parameters $P(C_i = k) = \phi_k$ and $X_i \mid C_i = k \sim \mathscr{N}(\mu_k, \Sigma_k)$:
$$p(x_i) = \sum_{k=1}^K \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)$$

This expression is called a **Gaussian mixture density** because it models a complex, multimodal overall probability distribution by blending (mixing) multiple individual Gaussian component distributions, weighted by their respective prior probabilities $\phi_k$.

---

### 2. Deriving the Posterior Cluster Probability (Responsibility)
Applying Bayes' rule for a specific observation $x_i$:
$$P(C_i = k \mid X_i = x_i) = \frac{p(X_i = x_i \mid C_i = k) P(C_i = k)}{p(x_i)}$$

Substituting the marginal density derived in Part 1 into the denominator:
$$P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^K \phi_j \mathscr{N}(x_i \mid \mu_j, \Sigma_j)} = \gamma_{ik}$$

The value $\gamma_{ik}$ represents the **posterior probability** that data point $x_i$ was generated by cluster $k$. It updates our prior belief $\phi_k$ by conditioning on the empirical evidence provided by the location of $x_i$.

---

### 3. One-Hot Encoding of the Latent Cluster Variable
Since $Z_{ik}$ is a binary indicator variable ($Z_{ik} \in \{0, 1\}$), its conditional expectation is equal to the probability of it being active:
$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = 1 \cdot P(Z_{ik} = 1 \mid X_i = x_i) + 0 \cdot P(Z_{ik} = 0 \mid X_i = x_i)$$
$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$

Extending this to the full vector $\mathbf{Z}_i$:
$$\mathbb{E}[\mathbf{Z}_i \mid X_i = x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i = x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i = x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

Thus, the soft cluster assignment vector in a GMM is exactly the mathematical conditional expectation $\mathbb{E}[\mathbf{Z}_i \mid X_i = x_i]$ of the latent identity structure.

---

### 4. From Soft Assignment to Hard Clustering
* **Soft Clustering:** Assigns a data point fractional membership weights across *all* clusters simultaneously via the probability vector $\boldsymbol{\gamma}_i$. This captures group overlap and preserves information about assignment ambiguity.
* **Hard Clustering:** Assigns each point to exactly one cluster by applying a deterministic decision rule ($\widehat{C}_i = \operatorname{arg\,max}_k \gamma_{ik}$), converting a continuous probability distribution into a discrete classification.

---

### 5. Conditional Expectation of the Observation Given the Cluster
Given $X_i \mid C_i = k \sim \mathscr{N}(\mu_k, \Sigma_k)$, the standard properties of a Gaussian distribution give:
$$\mathbb{E}[X_i \mid C_i = k] = \mu_k$$

* **Interpretation of $\mu_k$:** Represents the spatial center of gravity for component $k$.
* **Comparison:** $\mathbb{E}[\mathbf{Z}_i \mid X_i = x_i]$ operates in the **latent space** $[0, 1]^K$, mapping a known observation to its cluster membership probabilities. Conversely, $\mathbb{E}[X_i \mid C_i = k]$ operates in the **feature space** $\mathbb{R}^d$, mapping a specific cluster identity to its expected physical location.

---

### 6. The Complete-Data Likelihood
Given the complete-data likelihood equation:
$$p(\mathbf{x}, \mathbf{z}) = \prod_{i=1}^n \prod_{k=1}^K \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

Taking the natural logarithm transforms the products into sums:
$$\ell_c = \log p(\mathbf{x}, \mathbf{z}) = \sum_{i=1}^n \sum_{k=1}^K \log \left( \left[ \phi_k \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$
$$\ell_c = \sum_{i=1}^n \sum_{k=1}^K z_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

If the indicator variables $z_{ik}$ were known, this expression would decouple across clusters. This would allow us to optimize the parameters ($\mu_k, \Sigma_k$) for each cluster independently using standard, closed-form Maximum Likelihood Estimation (MLE) equations, completely avoiding the need for an iterative optimization loop.

---

### 7. The EM Interpretation
The expected complete-data log-likelihood $Q$ is formed by taking the expectation of $\ell_c$ with respect to the conditional distribution of the unobserved latent variables, given the data and current parameter updates:
$$Q = \mathbb{E}_{\mathbf{Z} \mid \mathbf{X}, \boldsymbol{\theta}^t}[\ell_c] = \sum_{i=1}^n \sum_{k=1}^K \mathbb{E}[Z_{ik} \mid X_i = x_i] \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

Substituting $\mathbb{E}[Z_{ik} \mid X_i = x_i] = \gamma_{ik}$ yields:
$$Q = \sum_{i=1}^n \sum_{k=1}^K \gamma_{ik} \left[ \log \phi_k + \log \mathscr{N}(x_i \mid \mu_k, \Sigma_k) \right]$$

The E-step can be interpreted as a **conditional update** because it recalculates the cluster membership weights for each point, leveraging the current parameter estimates to update our beliefs about the latent variables.

---

### 8. Parameter Updates
To maximize $Q$ with respect to $\mu_k$, we isolate the terms in $Q$ that depend on $\mu_k$:
$$Q(\mu_k) = \sum_{i=1}^n \gamma_{ik} \left( -\frac{1}{2}(x_i - \mu_k)^T \Sigma_k^{-1} (x_i - \mu_k) \right) + \text{constant}$$

Taking the vector derivative with respect to $\mu_k$ and setting it to $\mathbf{0}$:
$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^n \gamma_{ik} \Sigma_k^{-1} (x_i - \mu_k) = \mathbf{0}$$
$$\sum_{i=1}^n \gamma_{ik} x_i = \sum_{i=1}^n \gamma_{ik} \mu_k \implies \mu_k^{\text{new}} = \frac{\sum_{i=1}^n \gamma_{ik} x_i}{\sum_{i=1}^n \gamma_{ik}} = \frac{1}{N_k} \sum_{i=1}^n \gamma_{ik} x_i$$

A similar optimization under the constraint $\sum \phi_k = 1$ using Lagrange multipliers yields the updates for $\Sigma_k^{\text{new}}$ and $\phi_k^{\text{new}}$. The responsibility $\gamma_{ik}$ acts as a continuous **fractional membership weight**, meaning every data point contributes partially to the parameter updates of every cluster based on its assignment probability.

---

### 9. Summary Interpretation
Gaussian Mixture Model (GMM) clustering can be viewed as an iterative process of conditional updating. The mixture weight $\phi_k$ represents the prior probability of belonging to cluster $k$, while the Gaussian density $\mathscr{N}(x_i \mid \mu_k, \Sigma_k)$ measures how compatible an observation $x_i$ is with that cluster. By applying Bayes' rule, the algorithm combines these elements to compute the responsibility $\gamma_{ik}$, which is the posterior probability of cluster membership after observing $x_i$. These responsibilities form the soft assignment expectation vector $\mathbb{E}[\mathbf{Z}_i \mid X_i = x_i]$. During the M-step, the algorithm updates the geometric parameters ($\phi_k, \mu_k, \Sigma_k$) using these posterior probabilities as fractional weights. This confirms that GMM clustering is fundamentally a form of probabilistic clustering driven by conditional expectations of latent variables.

---

### 10. Computational Simulation Contour Interpretation
The continuous background contour map visualizes the maximum value of the posterior responsibility vector $\max_k \gamma_{ik}$ at every point on the coordinate grid. Near the cluster centers ($\mu_k$), the responsibility values are close to $1.0$, creating solid color zones in the visualization. As you move away from the centers and cross the decision boundaries, the continuous color transition displays the blending of probabilities. This gradient directly illustrates the soft assignment expectation vector $\mathbb{E}[\mathbf{Z}_i \mid X_i = x_{\text{grid}}]$ derived in Part 3, highlighting regions of ambiguity where a point shares membership across multiple components.

In [9]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

# 1. Pipeline Implementation Class
class GMMFinancialSegmenter:
    def __init__(self, n_components=3):
        self.model = GaussianMixture(n_components=n_components, random_state=42)
        self.scaler = StandardScaler()

    def prepare_data(self, df, features):
        data = df[features].dropna()
        X = data.values
        X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        return X_train_scaled, X_test_scaled

    def fit(self, X_train):
        self.model.fit(X_train)
        print(f"Convergence status: {self.model.converged_}")
        print(f"Iterations required: {self.model.n_iter_}")

    def evaluate(self, X_test):
        score = self.model.score(X_test)
        print(f"Average out-of-sample log-likelihood: {score:.4f}")

    def plot_density_heatmap(self, X_train):
        fig = px.density_heatmap(x=X_train[:, 0], y=X_train[:, 1], marginal_x="histogram", marginal_y="histogram")
        fig.update_layout(title="Empirical 2D Density Heatmap (Training Data)", xaxis_title="Feature 1 (Scaled)", yaxis_title="Feature 2 (Scaled)", template="plotly_white")
        return fig

    def plot_assignments(self, X_data, title_text):
        x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
        y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
        grid_points = np.c_[xx.ravel(), yy.ravel()]

        responsibilities = self.model.predict_proba(grid_points)
        max_resp = responsibilities.max(axis=1).reshape(xx.shape)
        data_preds = self.model.predict(X_data)

        fig = go.Figure()
        fig.add_trace(go.Contour(x=np.linspace(x_min, x_max, 200), y=np.linspace(y_min, y_max, 200), z=max_resp, colorscale='Viridis', opacity=0.4, showscale=True, name="Max Responsibility"))
        fig.add_trace(go.Scatter(x=X_data[:, 0], y=X_data[:, 1], mode='markers', marker=dict(color=data_preds, colorscale='Portland', size=5), name="Data Instances"))

        fig.update_layout(title=title_text, xaxis_title="Feature 1 (Scaled)", yaxis_title="Feature 2 (Scaled)", template="plotly_white")
        return fig

# 2. Execution Script Framework
if __name__ == "__main__":
    n_samples = 1000
    mock_data = {
        'PURCHASES': np.concatenate([np.random.normal(500, 200, 400), np.random.normal(2500, 600, 300), np.random.normal(100, 50, 300)]),
        'CREDIT_LIMIT': np.concatenate([np.random.normal(2000, 500, 400), np.random.normal(7000, 1500, 300), np.random.normal(1000, 300, 300)])
    }
    df = pd.DataFrame(mock_data)

    segmenter = GMMFinancialSegmenter(n_components=3)
    target_features = ['PURCHASES', 'CREDIT_LIMIT']

    X_tr, X_te = segmenter.prepare_data(df, target_features)
    segmenter.fit(X_tr)
    segmenter.evaluate(X_te)

    fig_heatmap = segmenter.plot_density_heatmap(X_tr)
    fig_train = segmenter.plot_assignments(X_tr, "Training Assignment Plot Overlay")
    fig_test = segmenter.plot_assignments(X_te, "Out-of-Sample Test Assignment Plot Overlay")

    fig_heatmap.show()
    fig_train.show()
    fig_test.show()s

Convergence status: True
Iterations required: 4
Average out-of-sample log-likelihood: -0.8579
